# AtmosSim: Starter Notebook & Regime Validation

Welcome to **AtmosSim** — a physics-based atmospheric dispersion simulator and machine-learning benchmark for urban air quality.

This notebook provides an end-to-end walkthrough using the real **2018–2024 Multi-City Continuous Dataset** (245,472 hourly records across Delhi, Mumbai, Pune, and Bengaluru).

---

## 1. Load the Data

We load the consolidated multi-city continuous master dataset (`multicity_2018_2024_continuous_master.parquet`). Each record contains hourly ERA5 meteorology, cyclic temporal features, micro-to-mesoscale local line dispersion, synoptic CAMS regional background, and the coupled total PM2.5 target.

> **Important Boundary Disclosure (2022–2024 vs. 2018–2021 Backcast)**:  
> - **2022–2024**: Evaluated directly against real OpenAQ / CPCB reference ground monitoring stations across 19 historical dates covering all 4 seasons.  
> - **2018–2021**: Generated by driving the calibrated physics engine with historical Open-Meteo ERA5 meteorology and CAMS reanalysis. This portion has **not** been independently spot-checked against real 2018–2021 monitoring stations and must be treated as a physics-based model backcast.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Locate master parquet dataset
dataset_candidates = [
    Path("artifacts/multiyear_dataset/multicity_2018_2024_continuous_master.parquet"),
    Path("../artifacts/multiyear_dataset/multicity_2018_2024_continuous_master.parquet"),
    Path("multicity_2018_2024_continuous_master.parquet"),
]

dataset_path = next((p for p in dataset_candidates if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find multicity_2018_2024_continuous_master.parquet. Please check your data directory.")

print(f"Loading master dataset from: {dataset_path}")
df = pd.read_parquet(dataset_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])

cities_list = list(df["city"].unique())
min_date = df["timestamp"].min().strftime("%Y-%m-%d")
max_date = df["timestamp"].max().strftime("%Y-%m-%d")

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Cities:        {cities_list}")
print(f"Date Range:    {min_date} to {max_date}")
print(f"Total Cells:   {df.shape[0] * df.shape[1]:,} data points")

df.head(5)

---

## 2. Reproduce the Per-Regime Validation Table

Rather than relying on a single aggregate correlation figure that can mask regime-specific dynamics, AtmosSim is evaluated **strictly by atmospheric regime**.

Here we load the exact 19 real historical validation dates (from `artifacts/validation_dates.csv` and `docs/phase4_validation_report.md`), compute the Pearson $r$ and mean absolute error (MAE) live in the notebook, and assert that the values match the documented project audit.

In [ ]:
# Load exact 19 historical ground-truth validation records
val_csv_candidates = [
    Path("artifacts/validation_dates.csv"),
    Path("../artifacts/validation_dates.csv"),
    Path("validation_dates.csv"),
]
val_path = next((p for p in val_csv_candidates if p.exists()), None)

if val_path is not None:
    val_df = pd.read_csv(val_path)
else:
    # Exact table from docs/phase4_validation_report.md
    val_data = [
        {"date": "2022-11-04", "regime": "stubble_burning", "observed_pm25": 313.0, "simulated_total_pm25": 387.9},
        {"date": "2022-11-08", "regime": "stubble_burning", "observed_pm25": 345.0, "simulated_total_pm25": 392.3},
        {"date": "2023-11-03", "regime": "stubble_burning", "observed_pm25": 319.0, "simulated_total_pm25": 322.2},
        {"date": "2023-11-13", "regime": "stubble_burning", "observed_pm25": 286.0, "simulated_total_pm25": 309.7},
        {"date": "2024-11-18", "regime": "stubble_burning", "observed_pm25": 665.0, "simulated_total_pm25": 560.2},
        {"date": "2023-11-20", "regime": "stubble_burning", "observed_pm25": 390.0, "simulated_total_pm25": 383.3},
        {"date": "2024-11-15", "regime": "stubble_burning", "observed_pm25": 480.0, "simulated_total_pm25": 482.6},
        {"date": "2023-01-10", "regime": "winter_fog_inversion", "observed_pm25": 380.0, "simulated_total_pm25": 365.4},
        {"date": "2023-01-20", "regime": "winter_fog_inversion", "observed_pm25": 275.0, "simulated_total_pm25": 259.7},
        {"date": "2024-01-14", "regime": "winter_fog_inversion", "observed_pm25": 396.0, "simulated_total_pm25": 215.1},
        {"date": "2024-01-22", "regime": "winter_fog_inversion", "observed_pm25": 350.0, "simulated_total_pm25": 355.8},
        {"date": "2024-01-28", "regime": "winter_fog_inversion", "observed_pm25": 290.0, "simulated_total_pm25": 269.6},
        {"date": "2023-03-15", "regime": "moderate_transition", "observed_pm25": 115.0, "simulated_total_pm25": 87.4},
        {"date": "2023-04-10", "regime": "moderate_transition", "observed_pm25": 95.0, "simulated_total_pm25": 74.7},
        {"date": "2024-03-01", "regime": "moderate_transition", "observed_pm25": 135.0, "simulated_total_pm25": 113.8},
        {"date": "2024-04-20", "regime": "moderate_transition", "observed_pm25": 88.0, "simulated_total_pm25": 66.6},
        {"date": "2023-07-15", "regime": "south_asian_monsoon", "observed_pm25": 35.0, "simulated_total_pm25": 52.8},
        {"date": "2023-08-20", "regime": "south_asian_monsoon", "observed_pm25": 42.0, "simulated_total_pm25": 57.7},
        {"date": "2024-09-20", "regime": "south_asian_monsoon", "observed_pm25": 40.0, "simulated_total_pm25": 62.1},
    ]
    val_df = pd.DataFrame(val_data)

# Compute live per-regime metrics
regimes = [
    ("Stubble Burning Crisis", val_df["regime"].str.contains("stubble")),
    ("Moderate Spring / Summer", val_df["regime"].str.contains("moderate")),
    ("Winter Fog & Inversion", val_df["regime"].str.contains("fog")),
    ("Monsoon Washout", val_df["regime"].str.contains("monsoon")),
    ("All 19 Validation Dates", np.ones(len(val_df), dtype=bool)),
]

print("=========================================================================================")
print("                   REPRODUCED PER-REGIME GROUND-TRUTH VALIDATION TABLE                  ")
print("=========================================================================================")
print(f"{'Regime':<26} | {'N':>3} | {'Obs Range':>15} | {'Pearson r':>10} | {'MAE (ug/m3)':>12} | {'MAPE (%)':>9}")
print("-----------------------------------------------------------------------------------------")

results_dict = {}
for name, mask in regimes:
    sub = val_df[mask]
    obs = sub["observed_pm25"].values
    sim = sub["simulated_total_pm25"].values
    n = len(sub)
    obs_range = f"{obs.min():.0f} - {obs.max():.0f}"
    
    r = stats.pearsonr(obs, sim)[0] if n > 2 else np.nan
    mae = np.mean(np.abs(obs - sim))
    mape = np.mean(np.abs(obs - sim) / obs) * 100
    
    results_dict[name] = {"r": r, "mae": mae, "mape": mape}
    r_str = f"{r:+.3f}" if not np.isnan(r) else "N/A"
    if name == "Monsoon Washout":
        r_str = f"{r_str}*"
    print(f"{name:<26} | {n:3d} | {obs_range:>15} | {r_str:>10} | {mae:12.1f} | {mape:8.1f}%")

print("=========================================================================================")
print("* Note on Monsoon: Pearson r = +0.715 is computed across a very narrow 7 ug/m3 range (N=3)")
print("  and is statistically fragile; CAMS regional background floor maintains ~50 ug/m3 bias.")
print()

# Assertions matching documented validation report
assert np.isclose(results_dict["Stubble Burning Crisis"]["r"], 0.950, atol=0.01), "Stubble r mismatch"
assert np.isclose(results_dict["Moderate Spring / Summer"]["r"], 0.988, atol=0.01), "Moderate r mismatch"
assert np.isclose(results_dict["Winter Fog & Inversion"]["r"], 0.179, atol=0.01), "Fog r mismatch"
assert np.isclose(results_dict["Monsoon Washout"]["r"], 0.715, atol=0.01), "Monsoon r mismatch"
assert np.isclose(results_dict["All 19 Validation Dates"]["r"], 0.948, atol=0.01), "All dates r mismatch"
print("LIVE VERIFICATION PASSED: Recomputed Pearson correlations match documented report exactly.")

---

## 3. Local vs. Regional Decomposition (November 18, 2024 Event)

Delhi's most catastrophic pollution days (e.g. November 18, 2024, when ground stations recorded $665\ \mu\text{g/m}^3$) are dominated by regional transboundary agricultural stubble smoke advected from Punjab and Haryana across hundreds of kilometers.

Because micro-to-mesoscale dispersion models simulate local road traffic emissions ($C_{\text{local}}$), attempting to predict total ambient PM2.5 without an external boundary condition leads to negative rank correlations on regional crisis days. AtmosSim explicitly decomposes ambient concentration:
$$C_{\text{total}}(t) = C_{\text{local}}(t) + C_{\text{regional}}(t)$$

Below, we filter to November 18, 2024 and plot this decomposition over that day's 24 hours alongside the real observed ground station level.

> **Mechanism Note on the Nov 18 Peak Gap**:  
> While the coupled model captures the extreme crisis scale ($560.2\ \mu\text{g/m}^3$ simulated vs. $665.0\ \mu\text{g/m}^3$ observed), the residual underprediction is driven by **CAMS spatial grid resolution ($\sim 40\text{ km}$ / $0.25^\circ \times 0.25^\circ$)**, which spatially averages dense, narrow transboundary agricultural smoke plumes. (In contrast, unmodeled secondary aqueous sulfate chemistry is the distinct limitation identified specifically for the *winter fog regime*, not this stubble crisis).

In [ ]:
# Filter to Delhi on Nov 18, 2024
delhi_nov18 = df[(df["city"] == "Delhi") & (df["timestamp"].dt.strftime("%Y-%m-%d") == "2024-11-18")].sort_values("timestamp")

hours = delhi_nov18["timestamp"].dt.hour
c_local = delhi_nov18["local_dispersion_pm25"].values
c_reg = delhi_nov18["cams_regional_pm25"].values
c_total = delhi_nov18["target_pm25"].values
real_observed_24h = 665.0  # OpenAQ ground monitor record

plt.figure(figsize=(12, 5.5), dpi=150)

# Stacked area plot: Regional baseline at bottom, local road plume stacked on top
plt.fill_between(hours, 0, c_reg, color="#3498db", alpha=0.65, label=r"CAMS Synoptic Regional Background ($C_{\mathrm{regional}}$)")
plt.fill_between(hours, c_reg, c_reg + c_local, color="#e74c3c", alpha=0.75, label=r"CALINE4 Local Road Network Dispersion ($C_{\mathrm{local}}$)")
plt.plot(hours, c_total, color="#2c3e50", linewidth=2.5, label=r"Coupled Total Simulated PM2.5 ($C_{\mathrm{total}}$)")

# Observed 24h benchmark line
plt.axhline(real_observed_24h, color="#8e44ad", linestyle="--", linewidth=2.0, label=f"OpenAQ Ground Observed 24h Mean ({real_observed_24h:.0f} µg/m³)")

plt.title("AtmosSim Source Decomposition — Delhi Severe Smog Episode (Nov 18, 2024)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Hour of Day (UTC)", fontsize=11, fontweight="semibold")
plt.ylabel("PM2.5 Concentration (µg/m³)", fontsize=11, fontweight="semibold")
plt.xticks(range(0, 24, 2), [f"{h:02d}:00" for h in range(0, 24, 2)])
plt.ylim(0, 750)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper right", framealpha=0.9, fontsize=10)
plt.tight_layout()
plt.show()

print(f"Nov 18 2024 Simulated 24h Mean: {c_total.mean():.1f} ug/m3 (Regional: {c_reg.mean():.1f} + Local: {c_local.mean():.1f})")
error_val = c_total.mean() - real_observed_24h
rel_err = (error_val / real_observed_24h) * 100
print(f"Nov 18 2024 Real Observed 24h:   {real_observed_24h:.1f} ug/m3 (Error: {error_val:+.1f} ug/m3, {rel_err:+.1f}%)")

---

## 4. Machine Learning Baseline Model End-to-End

We train a linear Ridge regression baseline predicting `target_pm25` from purely operational features (meteorology, diurnal/seasonal time encodings, and city indicators).

> **No Data Leakage**: Target columns, local dispersion outputs, and simulator internal states are strictly excluded from the feature matrix. We evaluate on a chronological out-of-sample holdout (Train: 2018–2022, Test: 2023–2024).

In [ ]:
# Operational feature subset
MET_FEATURES = [
    "wind_speed_mps", "wind_u_mps", "wind_v_mps", "temperature_c",
    "relative_humidity_pct", "surface_pressure_hpa", "boundary_layer_height_m",
    "direct_normal_irradiance_w_m2", "cloud_cover_pct", "hour", "day_of_week",
    "month", "is_weekend", "sin_hour", "cos_hour", "sin_month", "cos_month",
    "cams_regional_pm25"
]

# One-hot encode cities
df_encoded = pd.get_dummies(df, columns=["city"], drop_first=False)
city_cols = [c for c in df_encoded.columns if c.startswith("city_")]
features = MET_FEATURES + city_cols

# Strict chronological split
df_encoded["year"] = df_encoded["timestamp"].dt.year
train_mask = df_encoded["year"] <= 2022
test_mask = df_encoded["year"] >= 2023

X_train = df_encoded.loc[train_mask, features]
y_train = df_encoded.loc[train_mask, "target_pm25"].values

X_test = df_encoded.loc[test_mask, features]
y_test = df_encoded.loc[test_mask, "target_pm25"].values

print(f"Train Split (2018-2022): {len(X_train):,} samples")
print(f"Test Split  (2023-2024): {len(X_test):,} samples")

# Standardize and fit Ridge baseline
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)

y_pred = ridge.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n--- Ridge Baseline Evaluation (2023-2024 Holdout) ---")
print(f"MAE:  {mae:.2f} ug/m3")
print(f"RMSE: {rmse:.2f} ug/m3")
print(f"R2:   {r2:.3f}")

> **Note on ML Performance**:  
> Ridge regression is a linear baseline ($R^2 \approx 0.60$). In the full AtmosSim benchmark suite, gradient-boosted tree models (e.g. **XGBoost achieved $R^2 = 0.911$**, Random Forest $R^2 = 0.884$) capture the strong non-linear interactions inherent in atmospheric boundary layer physics ($1/\text{PBLH} \times 1/u$).  
> For the full 7-model benchmark comparison and training scripts, see [`scripts/run_multiyear_ml_benchmark.py`](https://github.com/Suraj-codes1410/AtmosSim/blob/main/scripts/run_multiyear_ml_benchmark.py).

---

## 5. Where to Learn More & Documented Limitations

### Project Links
- **GitHub Repository**: [Suraj-codes1410/AtmosSim](https://github.com/Suraj-codes1410/AtmosSim)
- **Phase 4 Validation Report**: [`docs/phase4_validation_report.md`](https://github.com/Suraj-codes1410/AtmosSim/blob/main/docs/phase4_validation_report.md)
- **Project Verification Report**: [`docs/project_verification_report.md`](https://github.com/Suraj-codes1410/AtmosSim/blob/main/docs/project_verification_report.md)

### Known Model Limitations (Recap)
1. **Winter Fog Secondary Chemistry ($r = 0.179$)**: AtmosSim models primary physical dispersion and assimilates regional CAMS background; it does not simulate aqueous sulfate/nitrate oxidation chemistry inside dense liquid water fog.
2. **Monsoon Regional Floor**: CAMS reanalysis maintains an elevated baseline floor ($\sim 35-55\ \mu\text{g/m}^3$) over India during active monsoon washouts, leading to overprediction on pristine rainout days.
3. **2018–2021 Backcast Boundary**: Observational validation against OpenAQ stations was conducted on 2022–2024; the 2018–2021 portion is a physics-based backcast.